In [1]:
import numpy as np


In [2]:
# Load the preprocessed dataset (400 timesteps)
DATA_PATH = "../data/processed/processed_400_timesteps.npz"

data = np.load(DATA_PATH, allow_pickle=True)

X = data["X"]
mask = data["mask"]
y = data["y"]
groups = data["groups"]

print("X shape:", X.shape)
print("y shape:", y.shape)


X shape: (10000, 400, 240)
y shape: (10000,)


### Why Feature Engineering?

Classical machine learning models such as Logistic Regression and Random Forest
cannot directly process time-series data.

Therefore, we convert each (400 × 240) time-series sample into a fixed-length
feature vector using statistical and temporal summary features.


In [3]:
def extract_features(X, mask):
    """
    Convert time-series data into fixed-length feature vectors.

    Parameters:
    X    : ndarray of shape (samples, timesteps, features)
    mask : ndarray of shape (samples, timesteps)

    Returns:
    feature_matrix : ndarray of shape (samples, engineered_features)
    """

    feature_list = []

    # Loop over each sample (bag)
    for i in range(X.shape[0]):

        # Get indices of valid timesteps using mask
        valid_idx = np.where(mask[i] == 1)[0]

        # Select only valid data
        X_valid = X[i, valid_idx, :]   # Shape: (valid_timesteps, 240)

        # --- Statistical Features ---
        mean_feat = np.mean(X_valid, axis=0)   # Mean over time
        std_feat  = np.std(X_valid, axis=0)    # Standard deviation
        max_feat  = np.max(X_valid, axis=0)    # Maximum value
        min_feat  = np.min(X_valid, axis=0)    # Minimum value

        # --- Temporal Change Features ---
        # Difference between last and first timestep
        diff_feat = X_valid[-1] - X_valid[0]

        # Concatenate all features into one vector
        sample_features = np.concatenate([
            mean_feat,
            std_feat,
            max_feat,
            min_feat,
            diff_feat
        ])

        feature_list.append(sample_features)

    # Convert list to NumPy array
    feature_matrix = np.array(feature_list)

    return feature_matrix


In [4]:
X_features = extract_features(X, mask)

print("Feature matrix shape:", X_features.shape)


Feature matrix shape: (10000, 1200)


In [5]:
# Check for NaNs or infinite values
print("NaN values:", np.isnan(X_features).sum())
print("Infinite values:", np.isinf(X_features).sum())


NaN values: 0
Infinite values: 0


In [6]:
# Save features for ML models
SAVE_PATH = "../data/processed/ml_features_400.npy"

np.save(SAVE_PATH, X_features)

print("ML features saved to:", SAVE_PATH)


ML features saved to: ../data/processed/ml_features_400.npy


### Feature Engineering Summary

- Input data shape: (10000, 400, 240)
- Feature extraction method:
  - Mean
  - Standard deviation
  - Maximum
  - Minimum
  - Temporal difference (last − first)
- Output feature matrix shape: (10000, 1200)

These features enable the use of classical machine learning models
for solar flare prediction.
